In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
llm_name: str = "Qwen/Qwen3-0.6B"
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(llm_name)

c:\Users\Charlie\anaconda3\envs\ai_assistant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0106 07:08:10.530000 41364 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
lora_r = 16
lora_alpha = 32
lora_dropout = 0.1

# llm = get_peft_model(llm, LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=lora_r,
#     lora_alpha=lora_alpha,
#     lora_dropout=lora_dropout,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     inference_mode=False,
# ))
llm = PeftModel.from_pretrained(llm, "checkpoints/base_llm_qwen3-0.6b_lora/with_embed-10000")
llm.print_trainable_parameters()

trainable params: 0 || all params: 606,142,464 || trainable%: 0.0000


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel

num_vectors = 32
class EmbeddingToQwenCrossAttentionModel(nn.Module):
    """
    Model that takes input text, encodes it with an embedding model,
    and projects it to 32 vectors of dimension equal to Qwen's hidden_dim
    using a cross-attention mechanism.
    """
    def __init__(self, 
                 embedding_model, 
                 embedding_dim: int,
                 qwen_hidden_dim: int, 
                 num_vectors: int = 32,
                 embed_tokenizer=None):
        super().__init__()
        self.embedding_model = embedding_model  # must take text and return (batch, embed_dim)
        self.embed_tokenizer = embed_tokenizer  # Tokenizer for embedding model
        self.qwen_hidden_dim = qwen_hidden_dim
        self.num_vectors = num_vectors

        # Linear to project embedding model's output to a "memory" for cross-attention
        self.memory_proj = nn.Linear(
            embedding_dim, qwen_hidden_dim
        )

        # Learnable queries to use for cross-attention
        self.queries = nn.Parameter(torch.randn(1, num_vectors, qwen_hidden_dim))

        # Cross-attention module: query = [num_vectors, hidden_dim], key & value = [seq, hidden_dim]
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=qwen_hidden_dim,
            num_heads=4,
            batch_first=True
        )

    def freeze_embedding_model(self):
        for param in self.embedding_model.parameters():
            param.requires_grad = False

    def forward(self, text_input):
        """
        text_input: Can be either:
            - List of strings (raw text)
            - Tensor of token IDs (must be from embedding model's tokenizer)
        Returns: projected_vectors: shape (batch, num_vectors, qwen_hidden_dim)
        """
        # 1. Encode text to embedding
        # If text_input is a tensor, assume it's already tokenized for embedding model
        # If it's strings or needs tokenization, tokenize with embedding model's tokenizer
        if isinstance(text_input, list) and isinstance(text_input[0], str):
            # Raw text - tokenize with embedding model's tokenizer
            if self.embed_tokenizer is None:
                raise ValueError("embed_tokenizer required when passing raw text")
            encoded = self.embed_tokenizer(
                text_input,
                padding=True,
                truncation=True,
                max_length=512,  # E5 models typically use 512 max length
                return_tensors="pt"
            )
            # Move to same device as embedding model
            device = next(self.embedding_model.parameters()).device
            embedding_input_ids = encoded["input_ids"].to(device)
            embedding = self.embedding_model(input_ids=embedding_input_ids)
        elif isinstance(text_input, torch.Tensor):
            # Assume it's already tokenized for embedding model
            embedding = self.embedding_model(input_ids=text_input)
        else:
            raise ValueError(f"Unsupported input type: {type(text_input)}")
        if hasattr(embedding, "last_hidden_state"):  # For HF models
            if len(embedding.last_hidden_state.shape) == 3:
                # Use [CLS] or mean pooling
                emb_vec = embedding.last_hidden_state.mean(dim=1)
            else:
                emb_vec = embedding.last_hidden_state
        elif isinstance(embedding, torch.Tensor):
            # (batch, embed_dim)
            emb_vec = embedding
        else:
            raise ValueError("Unknown embedding_model output structure")

        # 2. Project embedding to Qwen hidden dimension
        memory = self.memory_proj(emb_vec).unsqueeze(1)  # (batch, 1, qwen_hidden_dim)

        # 3. Prepare queries: expand learnable [1, num_vectors, h] to batch
        queries = self.queries.expand(emb_vec.shape[0], -1, -1)  # (batch, num_vectors, dim)

        # 4. Cross attention: query=(B, N, D), key/value=(B, 1, D)
        # nn.MultiheadAttention expects shape (batch_size, seq_length, embed_dim)
        attended, _ = self.cross_attn(queries, memory, memory)  # (batch, num_vectors, dim)
        return attended

    def load_projector(self, path):
        import os

        additional_path = os.path.join(path, "additional_components.pt")
        additional_components = torch.load(additional_path, map_location=device)
        self.memory_proj.load_state_dict(additional_components["memory_proj"])
        # queries is a Parameter, load it directly
        if "queries" in additional_components:
            self.queries.data.copy_(additional_components["queries"])
        self.cross_attn.load_state_dict(additional_components["cross_attn"])

embedding_model = "dwzhu/e5-base-4k"
embed_model = AutoModel.from_pretrained(embedding_model)
embed_model = PeftModel.from_pretrained(embed_model, "checkpoints/base_llm_qwen3-0.6b_lora/with_qwen_embed/embed_model", is_trainable=True)
embed_tokenizer = AutoTokenizer.from_pretrained(embedding_model)
cross_proj = EmbeddingToQwenCrossAttentionModel(
    embedding_model=embed_model,
    embedding_dim=embed_model.config.hidden_size,
    qwen_hidden_dim=llm.config.hidden_size,
    num_vectors=num_vectors,
    embed_tokenizer=embed_tokenizer
).to(device)
cross_proj.load_projector("checkpoints/base_llm_qwen3-0.6b_lora/with_embed")
cross_proj.freeze_embedding_model()
class QwenWithCrossAttention(nn.Module):
    def __init__(self, qwen_model, cross_proj):
        super().__init__()
        self.qwen_model = qwen_model
        self.cross_proj = cross_proj

    def forward(self, input_ids=None, attention_mask=None, labels=None, text=None):
        # Encode text with embedding model (needs raw text, not Qwen token IDs)
        if text is None:
            raise ValueError("text parameter required for embedding model")
        embeddings = self.cross_proj(text)  # Pass raw text, not input_ids
        embed_layer = self.qwen_model.get_input_embeddings()
        text_embeddings = embed_layer(input_ids)
        total_embeddings = torch.cat([text_embeddings, embeddings], dim=1)
        
        # Pass embeddings to Qwen model
        outputs = self.qwen_model(inputs_embeds=total_embeddings, attention_mask=attention_mask, labels=labels)
        
        return outputs

    def get_num_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model = QwenWithCrossAttention(llm, cross_proj)
print(model.get_num_trainable_parameters())

5018624


In [5]:
test_text = "Can you please tell me something about that movie BLOCKER we saw yesterday?"
qwen_tokenizer = AutoTokenizer.from_pretrained(llm_name)
input_ids = qwen_tokenizer(
    test_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
).to(device)
print(input_ids['attention_mask'])
print(input_ids['attention_mask'].shape)
output = model(text=[test_text], input_ids=torch.empty(1, 0, dtype=torch.long).to(device))
qwen_tokenizer.decode(output.logits.argmax(dim=-1)[0][-1])


tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')
torch.Size([1, 15])


'I'

In [10]:
test_text = "I really hope everything is alright over there :("
qwen_tokenizer = AutoTokenizer.from_pretrained(llm_name)
embed_tokens = cross_proj([test_text])
outputs = model.qwen_model(inputs_embeds=embed_tokens)
qwen_tokenizer.decode(outputs.logits.argmax(dim=-1)[0][-1])
embed_layer = model.qwen_model.get_input_embeddings()

def generate(text):
    new_text = "I"
    while True:
        embed_tokens = cross_proj([text])
        input_ids = qwen_tokenizer(
            new_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)
        input_tokens = embed_layer(input_ids["input_ids"])
        total_tokens = torch.cat([embed_tokens, input_tokens], dim=1)
        outputs = model.qwen_model(inputs_embeds=total_tokens)
        output_text = qwen_tokenizer.decode(outputs.logits.argmax(dim=-1)[0][-1])
        new_text += output_text
        print(new_text, flush=True)
        if output_text == "<|im_end|>":
            return new_text
generate(test_text)


I'm
I'm feeling
I'm feeling alright
I'm feeling alright,
I'm feeling alright, hope
I'm feeling alright, hope u
I'm feeling alright, hope u all
I'm feeling alright, hope u all ok
I'm feeling alright, hope u all ok over
I'm feeling alright, hope u all ok over there
I'm feeling alright, hope u all ok over there<|im_end|>


"I'm feeling alright, hope u all ok over there<|im_end|>"